# Week 2 — Derived Signals Dashboard
### Mini Hedge Fund | Phase 1

**What this notebook does:**  
Transforms raw economic numbers into the signals quant traders actually use.  
Raw CPI of 326.785 tells you nothing. This notebook shows you:
- How fast inflation is changing (MoM %)
- The annual trend (YoY %)
- How unusual current readings are (Z-scores)
- What the market *expected* vs what actually happened (Surprise)
- The trading signal that results from the surprise

**Instruments we are building toward trading:** TLT (Treasury bonds) and SPY (S&P 500)  
**Core logic:** Hot CPI surprise → yields rise → TLT falls | Cool surprise → TLT rises


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

# Repo root = folder containing mini_hedge/ (works from repo root or notebooks/)
_cwd = Path.cwd().resolve()
if (_cwd / "mini_hedge").is_dir():
    project_root = _cwd
elif (_cwd.parent / "mini_hedge").is_dir():
    project_root = _cwd.parent
else:
    _ex = (os.environ.get("MINI_HEDGE_ROOT") or "").strip()
    project_root = Path(_ex).resolve() if _ex else None
    if project_root is None or not (project_root / "mini_hedge").is_dir():
        raise RuntimeError(
            "Cannot find mini_hedge: open Jupyter with cwd = repo or notebooks/, "
            "or set env MINI_HEDGE_ROOT to the repo path."
        )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# Our modules
from mini_hedge.transforms import enrich_series, signals_snapshot, SERIES_MAP
from mini_hedge.surprises  import compute_surprises, describe_latest_surprise
from mini_hedge.prices     import load_10y_yield, yield_to_price_proxy, yield_around_releases

# Chart style
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family']       = 'DejaVu Sans'

BLUE   = '#2B579A'
RED    = '#C0392B'
GREEN  = '#27AE60'
ORANGE = '#E67E22'
GREY   = '#7F8C8D'

print('✓ Setup complete')

---
## Section 1 — Signals Snapshot
*The 'morning briefing' table — latest readings with all transforms applied.*


In [ ]:
# ── 1.1 Snapshot Table ────────────────────────────────────────────────────────
snap = signals_snapshot()

# Style the table
def color_zscore(val):
    """Colour-code Z-scores: red if extreme (|z| > 1.5), orange if elevated (|z| > 0.75)"""
    if pd.isna(val): return ''
    if   val >  1.5: return 'background-color: #FFDCDC; color: #C0392B; font-weight: bold'
    elif val >  0.75: return 'background-color: #FFF3CD'
    elif val < -1.5: return 'background-color: #D5F5E3; color: #27AE60; font-weight: bold'
    elif val < -0.75: return 'background-color: #EAF8F0'
    return ''

try:
    styled = snap.style.map(color_zscore, subset=["Z-Score"])
except AttributeError:
    styled = snap.style.applymap(color_zscore, subset=["Z-Score"])
print('Current readings across all four indicators:')
styled

---
## Section 2 — Reading the snapshot
Short **Δ%** → short-horizon change (MoM for CPI; 1-day for daily series like 10Y yield).
Long **Δ%** → long-horizon change (YoY for CPI).
**Z-Score** → how unusual the *level* is vs its own history (not the surprise yet).


---
## Section 3 — CPI surprise & trading signal
*Actual minus expected CPI; thresholded into hot / cool / flat.*


In [ ]:
# ── 3.1 Compute surprises ─────────────────────────────────────────────────────
# Real economist consensus from data/cpi_consensus.csv (survey / investing.com).
# For rolling benchmarks, use method="naive" or method="ema" in compute_surprises().

surprises = compute_surprises(
    method="real",
    surprise_threshold=0.15,
    consensus_csv="data/cpi_consensus.csv",
    auto_fetch_consensus=False,
)

# Show the latest 12 months
recent_surp = surprises.dropna(subset=['surprise']).tail(1)
print("Last 12 CPI releases — Actual vs Consensus vs Surprise:")
recent_surp[['date','mom_pct','consensus_mom','surprise','surprise_zscore','signal']].to_string(index=False)



In [ ]:
# ── 3.2 Latest Release Plain-English Narrative ────────────────────────────────
print(describe_latest_surprise(surprises))

In [ ]:

# ── 3.3 Surprise History Chart ────────────────────────────────────────────────
surp_plot = surprises[surprises['date'] >= '2015-01-01'].dropna(subset=['surprise'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# ── Top: MoM actual vs consensus ─────────────────────────────────────────────
ax1.plot(surp_plot['date'], surp_plot['mom_pct'],
         color=BLUE, linewidth=2, label='Actual MoM %')
ax1.plot(surp_plot['date'], surp_plot['consensus_mom'],
         color=ORANGE, linewidth=1.5, linestyle='--', alpha=0.8,
         label='Consensus (survey CSV)')
ax1.axhline(0, color=GREY, linewidth=0.8)
ax1.set_ylabel('MoM %')
ax1.set_title('CPI MoM: Actual vs Consensus', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)

# ── Bottom: The surprise bars ─────────────────────────────────────────────────
THRESHOLD = 0.15
colors2 = [RED if v > 0 else GREEN for v in surp_plot['surprise']]
ax2.bar(surp_plot['date'], surp_plot['surprise'],
        color=colors2, width=25, alpha=0.85)

ax2.axhline( THRESHOLD, color=RED,   linewidth=1.2, linestyle=':', alpha=0.7,
             label=f'+{THRESHOLD}% threshold (trade hot)')
ax2.axhline(-THRESHOLD, color=GREEN, linewidth=1.2, linestyle=':', alpha=0.7,
             label=f'-{THRESHOLD}% threshold (trade cool)')
ax2.axhline(0, color=GREY, linewidth=0.8)

# ── Value labels on bars that cross the threshold ─────────────────────────────
pad = 0.012   # gap between bar tip and label
for _, row in surp_plot.iterrows():
    val = row['surprise']
    if abs(val) < THRESHOLD:
        continue                # inside band — no label

    if val > 0:
        y_pos = val + pad
        va    = 'bottom'
        color = RED
    else:
        y_pos = val - pad
        va    = 'top'
        color = GREEN

    ax2.text(
        row['date'], y_pos,
        f'{val:+.2f}',
        fontsize=6.5,
        ha='center',
        va=va,
        color=color,
        fontweight='bold',
        rotation=90,
    )

ax2.set_ylabel('Surprise (Actual − Consensus)')
ax2.set_title('CPI Surprise — The Trading Signal', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

print("\nLEARNING NOTE:")
print("RED bar above +0.15% threshold → Signal to SHORT TLT (bonds fall on hot CPI)")
print("GREEN bar below -0.15% threshold → Signal to LONG TLT (bonds rise on cool CPI)")
print("Bars within the dotted lines → Too small to trade — stay out")
print("Numbers on bars = magnitude of the surprise in MoM %")


In [ ]:
# ── 3.4 Signal Distribution ───────────────────────────────────────────────────
all_signals = surprises.dropna(subset=['signal'])
hot   = (all_signals['signal'] ==  1).sum()
cool  = (all_signals['signal'] == -1).sum()
none_ = (all_signals['signal'] ==  0).sum()
total = len(all_signals)

print(f"{'='*50}")
print(f"  Signal Distribution ({total} releases total)")
print(f"  Consensus method: {surprises.attrs.get('surprise_method', '?')}")
print(f"{'='*50}")
print(f"  HOT  (+1):  {hot:3d}  ({hot/total*100:.1f}%)  → Trade: SHORT TLT")
print(f"  COOL (-1):  {cool:3d}  ({cool/total*100:.1f}%)  → Trade: LONG  TLT")
print(f"  NONE ( 0):  {none_:3d}  ({none_/total*100:.1f}%)  → No trade")
print(f"{'='*50}")
print(f"  Tradeable signals: {hot + cool} ({(hot+cool)/total*100:.1f}% of releases)")



---
## Section 4 — Bond Market Reaction (10Y Treasury Yield)
*Does the bond market move in the direction the surprise predicts? This is Week 3's question — previewed here.*


In [ ]:
# ── 4.1 10Y Yield around CPI release dates ───────────────────────────────────
yield_events = yield_around_releases(lookback_days=1, forward_days=5)

print(f"Yield data available for {len(yield_events)} CPI release dates")
print("\nMost recent 10 releases:")
print(yield_events.tail(10)[['release_date','yield_baseline','yield_day0',
                              'yield_chg_1d','yield_chg_5d','price_dir_1d']].to_string(index=False))
print("\nprice_dir_1d: +1 = bond prices UP (yield fell) | -1 = bond prices DOWN (yield rose)")

In [ ]:
# ── 4.2 Merge surprises with yield reactions ──────────────────────────────────
# CPI for reference month M is released in month M+1.
# We match on YEAR-MONTH period (not exact date) because the release falls
# anywhere from the 10th-15th of the month, not on the 1st.

surp_with_yield = surprises.copy()
yield_events_df = yield_around_releases()
yield_events_df['release_date'] = pd.to_datetime(yield_events_df['release_date'])

# Build a year-month key on both sides
surp_with_yield['ym'] = (
    surp_with_yield['date'] + pd.DateOffset(months=1)
).dt.to_period('M')

yield_events_df['ym'] = yield_events_df['release_date'].dt.to_period('M')

merged = surp_with_yield.merge(
    yield_events_df[['ym', 'release_date', 'yield_chg_1d', 'yield_chg_3d',
                     'yield_chg_5d', 'price_dir_1d']],
    on='ym', how='inner'
)

tradeable = merged[(merged['signal'] != 0) & merged['yield_chg_1d'].notna()].copy()

# Does the signal predict yield direction correctly?
# Hot  (+1): expect yield to RISE  → price_dir_1d should be -1 (bonds fell)
# Cool (-1): expect yield to FALL  → price_dir_1d should be +1 (bonds rose)
tradeable['signal_correct'] = tradeable.apply(
    lambda r: 1 if (r['signal'] ==  1 and r['price_dir_1d'] == -1) or
                   (r['signal'] == -1 and r['price_dir_1d'] ==  1) else 0,
    axis=1
)

# Win rate by signal type
hot_tr  = tradeable[tradeable['signal'] ==  1]
cool_tr = tradeable[tradeable['signal'] == -1]
wr_all  = tradeable['signal_correct'].mean() * 100
wr_hot  = hot_tr['signal_correct'].mean()  * 100 if len(hot_tr)  > 0 else 0
wr_cool = cool_tr['signal_correct'].mean() * 100 if len(cool_tr) > 0 else 0
n = len(tradeable)

print(f"\n{'='*56}")
print(f"  PRELIMINARY WIN RATE — CPI Surprise → Bond Direction")
print(f"  Survey CSV consensus (data/cpi_consensus.csv) | n = {n} tradeable events")
print(f"{'='*56}")
print(f"  Overall win rate:            {wr_all:.1f}%  ({tradeable['signal_correct'].sum()}/{n})")
print(f"  Hot signals  (short TLT):    {wr_hot:.1f}%  ({hot_tr['signal_correct'].sum()}/{len(hot_tr)})")
print(f"  Cool signals (long  TLT):    {wr_cool:.1f}%  ({cool_tr['signal_correct'].sum()}/{len(cool_tr)})")
print(f"{'='*56}")
print(f"\n  Target was 52–55%.  Preliminary result: {wr_all:.1f}%")
print(f"  ⚠ NOTE: Win rate depends on the consensus series. Compare naive / EMA / real")
print(f"    via python -m mini_hedge.surprises (or compare_consensus_methods).")
print(f"    A strong win rate is encouraging but should be checked across methods.")



In [ ]:
# ── 4.3 Average yield change by surprise direction ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Distribution of yield changes on CPI days
ax = axes[0]
hot_yields  = tradeable[tradeable['signal'] ==  1]['yield_chg_1d']
cool_yields = tradeable[tradeable['signal'] == -1]['yield_chg_1d']

ax.hist(hot_yields,  bins=20, color=RED,   alpha=0.6, label=f'Hot signals (n={len(hot_yields)})')
ax.hist(cool_yields, bins=20, color=GREEN, alpha=0.6, label=f'Cool signals (n={len(cool_yields)})')
ax.axvline(0, color=GREY, linewidth=1)
ax.set_title('10Y Yield Change on CPI Release Days', fontweight='bold')
ax.set_xlabel('Yield Change (percentage points)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# Right: Average yield change by signal type
ax2 = axes[1]
avgs = {
    'Hot (+1)': hot_yields.mean(),
    'Cool (-1)': cool_yields.mean(),
}
ax2.bar(avgs.keys(), avgs.values(), color=[RED, GREEN], alpha=0.7)
ax2.axhline(0, color=GREY, linewidth=1)
ax2.set_title('Mean 1D Yield Change by Signal', fontweight='bold')
ax2.set_ylabel('Avg yield chg (pp)')

plt.tight_layout()
plt.show()

---
## Section 5 — Context: Fed Rate vs CPI
*Understanding the relationship your signal is built on.*


In [ ]:
# ── 5.1 Fed Funds Rate vs CPI YoY — The Relationship ─────────────────────────
fed  = enrich_series('FEDFUNDS')[['date','value']].rename(columns={'value':'fed_rate'})
cpi2 = enrich_series('CUUR0000SA0')[['date','yoy_pct']].rename(columns={'yoy_pct':'cpi_yoy'})

combo = fed.merge(cpi2, on='date', how='inner')
combo = combo[combo['date'] >= '2000-01-01']

fig, ax = plt.subplots(figsize=(14, 5))
ax2 = ax.twinx()

ax.plot(combo['date'],  combo['cpi_yoy'],  color=BLUE,   linewidth=1.8, label='CPI YoY %  (left)')
ax2.plot(combo['date'], combo['fed_rate'], color=ORANGE, linewidth=2,   label='Fed Funds Rate (right)')

ax.axhline(2.0, color=BLUE, linewidth=1, linestyle='--', alpha=0.4)
ax.set_ylabel('CPI YoY %',     color=BLUE)
ax2.set_ylabel('Fed Rate %',   color=ORANGE)
ax.set_title('CPI vs Fed Funds (context)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## Section 6 — Path 3: VIX, MOVE & IV−RV

**Data:** `vol_indices` in SQLite (populate with `python scripts/bootstrap_db.py` or `python -m mini_hedge.cli fetch-vol`). **SPY/TLT** for realized vol: `yfinance` (same as `prices.fetch_yfinance`).

**Note:** `MOVE − scaled·RV(TLT)` uses a **rough visual scale** only — see `treasury_move_minus_scaled_rv` docstring in `transforms.py`.

In [ ]:
# ── 6.1 Load vol indices + SPY/TLT for RV ─────────────────────────────────────
from mini_hedge import storage
from mini_hedge.transforms import (
    realized_volatility,
    equity_iv_minus_rv,
    treasury_move_minus_scaled_rv,
)
from mini_hedge.prices import fetch_yfinance

vix = storage.query_vol_index("^VIX", last_n=4000)
move = storage.query_vol_index("^MOVE", last_n=4000)
spread_eq = spread_rates = None
if vix.empty or move.empty:
    print("vol_indices empty — run: python scripts/bootstrap_db.py  OR  python -m mini_hedge.cli fetch-vol")
else:
    print(f"^VIX rows: {len(vix)}  |  ^MOVE rows: {len(move)}")
    spy = tlt = None
    try:
        spy = fetch_yfinance("SPY", start="2007-01-01")
        tlt = fetch_yfinance("TLT", start="2007-01-01")
        spy_rv = realized_volatility(spy, window_days=20)
        tlt_rv = realized_volatility(tlt, window_days=20)
        spread_eq = equity_iv_minus_rv(vix, spy_rv)
        spread_rates = treasury_move_minus_scaled_rv(move, tlt_rv)
        print("IV−RV spreads computed (equity + treasury scaled).")
    except ImportError as e:
        print("yfinance missing:", e)
    except Exception as e:
        print("SPY/TLT or spread step:", e)


In [ ]:
# ── 6.2 VIX / MOVE levels + IV−RV spread (last ~8 years of overlap) ───────────
if vix.empty or move.empty:
    print("Skip charts — no vol index data.")
elif spread_eq is None or spread_rates is None:
    print("Skip charts — SPY/TLT or spreads not available (see 6.1 messages).")
else:
    tail_years = 8
    cut = spread_eq["date"].max() - pd.DateOffset(years=tail_years)
    v = vix[vix["date"] >= cut]
    m = move[move["date"] >= cut]
    se = spread_eq[spread_eq["date"] >= cut].dropna(subset=["vix_minus_rv_spy"])
    sr = spread_rates[spread_rates["date"] >= cut].dropna(subset=["move_minus_scaled_rv_tlt"])

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax0, ax1 = axes
    ax0.plot(v["date"], v["close"], color=BLUE, label="VIX")
    ax0.plot(m["date"], m["close"], color=ORANGE, label="MOVE", alpha=0.85)
    ax0.set_ylabel("Index level")
    ax0.legend(loc="upper left")
    ax0.set_title("Vol indices (VIX vs MOVE)")

    roll = se["vix_minus_rv_spy"].rolling(63, min_periods=21).mean()
    std = se["vix_minus_rv_spy"].rolling(252, min_periods=63).std()
    ax1.plot(se["date"], se["vix_minus_rv_spy"], color=GREY, alpha=0.5, label="VIX − RV(SPY)")
    ax1.plot(se["date"], roll, color=BLUE, label="63d mean (equity IV−RV)")
    ax1.fill_between(se["date"], roll - std, roll + std, color=BLUE, alpha=0.08)
    ax1.plot(sr["date"], sr["move_minus_scaled_rv_tlt"], color=ORANGE, alpha=0.7, label="MOVE − scaled RV(TLT)")
    ax1.axhline(0, color="black", linewidth=0.6, linestyle="--")
    ax1.set_ylabel("Spread (rough units)")
    ax1.legend(loc="upper left")
    ax1.set_title("IV − realized vol spreads (treasury curve is scaled — see transforms.py)")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()


In [ ]:
print("""
WEEK 2 COMPLETE ✓
══════════════════════════════════════════════════════════

WHAT YOU BUILT
──────────────
transforms.py   MoM%, YoY%, Z-score, moving averages, snapshot table
prices.py       10Y yield proxy, release calendar, CSV loader for TLT/SPY
surprises.py    Survey CSV + naive + EMA consensus, surprise calc, signals
This notebook   Full signal dashboard with 6 chart sections

WHAT THE DATA SHOWS (run Sections 3–4 for current numbers)
──────────────────────────────────────────────────────────
This notebook uses data/cpi_consensus.csv for “expected” CPI (Section 3.1).
Survey consensus can sit closer to the print than a 3M naive or EMA — so
surprises and signals often differ from naive/EMA (e.g. smaller |surprise|,
fewer trade signals). See describe_latest_surprise() output in Section 3.2.

PRELIMINARY WIN RATE
────────────────────
See Section 4.2 (live stats). Compare naive / EMA / survey CSV side by side:
  python -m mini_hedge.surprises

WHAT COMES NEXT — Week 3: Formal Event Study
─────────────────────────────────────────────
□ Test across all CPI release dates back to 2015 (not just recent)
□ Measure win rate at 1d, 3d, 5d horizons — which is most consistent?
□ Test whether large surprises (|Z| > 1.0) have higher win rates
□ Break results by rate environment (hiking cycle vs cutting cycle)
□ Estimate expected profit per trade after costs (bid-ask, slippage)
□ Answer definitively: is the edge real, or is it noise?

KEEP CONSENSUS DATA FRESH
─────────────────────────
Update Mini_Hedge/data/cpi_consensus.csv as new survey forecasts appear
(e.g. investing.com economic calendar — date + forecast columns).
══════════════════════════════════════════════════════════
""")